# UniqueShip data prep (Kaggle)

Runs the data checks for AcousticWise on the UniqueShip *12 Class - 5 Hours Each* bundle. See `DATA.md` in the repo for the dataset and checksums.

**Before you start**
1. In the right sidebar choose **Add Input** and attach the `uniqueship-12-class-5-hours` dataset.
2. Run the cells **one at a time, in order**. Do not use Run All: the extraction runs in the background and takes a while.
3. `/tmp` is wiped when the Kaggle session resets. If a later cell says `MISSING`, rerun section 1.

## 1. Extract the bundle

Joins the split zip into `/tmp/joined.zip`, unzips it to `/tmp/stage` (about 26 GB) and writes `DONE` to `/tmp/extract.log`. Safe to run twice: it will not start a second job.

In [ ]:
import os, subprocess
D = "/kaggle/input/uniqueship-12-class-5-hours"
log_text = open("/tmp/extract.log").read() if os.path.exists("/tmp/extract.log") else ""
running = subprocess.run("ps aux | grep -c '[z]ip -q -s 0'", shell=True, capture_output=True, text=True).stdout.strip() != "0"
if "DONE" in log_text and os.path.isdir("/tmp/stage"):
    print("Already extracted. Nothing to do.")
elif running:
    print("Extraction is already running. Do not start another; use the progress cell.")
elif not os.path.isdir(D):
    print("Dataset not attached at", D, "- add it via 'Add Input' in the notebook sidebar.")
else:
    os.makedirs("/tmp/parts", exist_ok=True)
    for src, dst in [("allclass_5hours.z01", "allclass_5hours.z01"), ("allclass_5hours.zip.part", "allclass_5hours.zip")]:
        if not os.path.lexists(f"/tmp/parts/{dst}"):
            os.symlink(f"{D}/{src}", f"/tmp/parts/{dst}")
    cmd = "cd /tmp && rm -rf stage joined.zip && zip -q -s 0 parts/allclass_5hours.zip --out joined.zip && unzip -q joined.zip -d stage && rm joined.zip && echo DONE"
    log = open("/tmp/extract.log", "w")
    p = subprocess.Popen(["sh", "-c", cmd], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print("started, pid", p.pid)

Progress: rerun this cell until the log shows `DONE`, `stage` is about 26 GB, and the last number is 0.

In [ ]:
import subprocess
print(subprocess.run("date; tail -3 /tmp/extract.log; ls -la /tmp/joined.zip 2>/dev/null; du -sh /tmp/stage 2>/dev/null; ps aux | grep -E 'zip|unzip' | grep -v grep | wc -l", shell=True, capture_output=True, text=True).stdout)

## 2. Vessels per class in all five folds

Distinct vessels (MMSI) and clips per class and split, plus clips per vessel. Resolves the map ticket *Count vessels per class in all five folds*.

In [ ]:
import os
import pandas as pd
root = globals().get("ROOT", "/tmp/stage/allclass_5hours")
if not os.path.isdir(root):
    print("MISSING:", root, "- the session was reset. Rerun the link, join and unzip cells first.")
else:
    V, C = {}, {}
    per_vessel = None
    for k in range(1, 6):
        for s in ("train", "validation", "test"):
            df = pd.read_csv(f"{root}/fold_{k}/metadata_5s_{s}.csv", usecols=["label", "MMSI"], low_memory=False)
            ships = df[df.MMSI != 0]
            V[(k, s)] = ships.groupby("label").MMSI.nunique()
            C[(k, s)] = df.groupby("label").size()
            if (k, s) == (1, "train"):
                per_vessel = ships.groupby(["label", "MMSI"]).size().groupby("label").agg(["min", "median", "max"])
    classes = sorted(set().union(*[set(x.index) for x in C.values()]))
    for name, T in (("distinct vessels", V), ("clips", C)):
        tab = pd.DataFrame(index=classes)
        for k in range(1, 6):
            parts = [T[(k, s)].reindex(classes).fillna(0).astype(int).astype(str) for s in ("train", "validation", "test")]
            tab[f"fold {k}"] = parts[0] + "/" + parts[1] + "/" + parts[2]
        print(f"\n{name} per class, shown as train/validation/test (background has no vessels):")
        print(tab.to_string())
    print("\nclips per vessel in fold 1 train (min, median, max):")
    print(per_vessel.to_string())

## 3. Spectrum check

Average power spectrum per class (up to 200 clips each, fold 1 train), summarised in four bands up to the 16 kHz Nyquist of the 32 kHz audio. Shows whether there is vessel-related energy above 8 kHz, which decides whether resampling to 16 kHz is safe. Resolves the map ticket *Check spectrum on Kaggle data*.

In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import welch
root = globals().get("ROOT", "/tmp/stage/allclass_5hours")
N = globals().get("N_PER_CLASS", 200)
if not os.path.isdir(root):
    print("MISSING:", root, "- run the extraction section first.")
else:
    bands = [(0, 1000), (1000, 4000), (4000, 8000), (8000, 16000)]
    names = [f"{lo // 1000}-{hi // 1000} kHz" for lo, hi in bands]
    meta = pd.read_csv(f"{root}/fold_1/metadata_5s_train.csv", usecols=["label", "path"], low_memory=False)
    rows = {}
    for label, g in meta.groupby("label"):
        sample = g.sample(min(N, len(g)), random_state=0)
        psds = []
        for p in sample.path:
            x, sr = sf.read(os.path.join(root, "fold_1", p))
            f, pxx = welch(x, fs=sr, nperseg=4096)
            psds.append(pxx)
        mean_psd = np.mean(psds, axis=0)
        rows[label] = [mean_psd[(f >= lo) & (f < hi)].sum() for lo, hi in bands]
    power = pd.DataFrame(rows, index=names).T
    share = (power.div(power.sum(axis=1), axis=0) * 100).round(2)
    above_bg = (10 * np.log10(power.div(power.loc["background"], axis=1))).round(1)
    print(f"Sampled up to {N} clips per class (fold 1 train), sample rate {sr} Hz.")
    print("\nShare of total power per band, % (each row sums to 100):")
    print(share.to_string())
    print("\nMean power per band relative to the background class, dB (0 = same as background):")
    print(above_bg.to_string())